# Notebook 03: Data Preprocessing & Feature Engineering

## Deskripsi Singkat
Tahap ini adalah proses **"pemasakan"** data mentah yang sudah bersih secara kualitas (dari tahap EDA) menjadi format yang siap dikonsumsi oleh algoritma *Machine Learning*. Fokus utama kita adalah menjaga **integritas data**, melakukan validasi, dan menghindari **Data Leakage**.

---

## Alur Kerja (Workflow)

1. **Load Interim Data**: Memanggil data hasil pembersihan dari folder `data/interim/`.

2. **Data Validation**: Pengecekan kualitas data untuk memastikan skema data sesuai dengan standar sebelum masuk ke proses modeling.

3. **Data Splitting (Train-Valid-Test)**: Membagi data menjadi tiga bagian utama:
   * **Data Train (80%)**: Digunakan untuk melatih model.
   * **Data Validation (10%)**: Diambil dari porsi data uji untuk optimasi dan *tuning* model (mencegah *overfitting*).
   * **Data Test (10%)**: Data rahasia yang benar-benar baru untuk evaluasi akhir performa model.

4. **Handling Missing Values (Imputation)**: Mengisi nilai kosong (`NaN`) menggunakan statistik (*Mean/Median/Mode*) berdasarkan referensi dari Data Train.

5. **Feature Engineering**:
   * **Encoding**: Transformasi data teks/kategorikal menjadi numerik.
   * **Scaling**: Penyeragaman skala fitur numerik agar performa model lebih stabil.

6. **Data Export**: Menyimpan hasil akhir (`Train`, `Valid`, dan `Test`) ke folder `data/processed/`.

---

## Kesimpulan Tahap Ini
Seluruh proses pemrosesan data dilakukan secara terpisah antara data Train, Valid, dan Test untuk menjamin objektivitas model. Hasil akhir berupa data dalam kondisi **"Model-Ready"** yang siap digunakan pada tahap **Notebook 04: Modeling**.

### 1. Blok Import & Setup Logging

In [1]:
import pandas as pd
import sys
import joblib
import logging
import numpy as np
from sklearn.model_selection import train_test_split
# TAMBAHKAN INI:
from sklearn.preprocessing import StandardScaler, LabelEncoder 
import warnings
warnings.filterwarnings('ignore')

# Supaya notebook bisa baca folder src
sys.path.append('../')
from src import utils

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info("Setup library dan logging (dengan StandardScaler) selesai.")

2026-02-17 19:28:12,726 - INFO - Setup library dan logging (dengan StandardScaler) selesai.


### 2. Load Data & Validasi Singkat

In [2]:
import joblib
import os

def load_and_validate():
    """
    Memuat data interim dalam format .pkl berdasarkan config
    dan melakukan validasi awal.
    """
    # 1. Load Config untuk ambil nama file
    cfg = utils.load_config('../config/config.yaml')
    path_interim = os.path.join('../', cfg['paths']['interim_dataset_dir'], cfg['paths']['interim_filename'])
    
    try:
        # 2. Load file .pkl (Karena data kamu data_gabungan.pkl)
        df_interim = joblib.load(path_interim)
        logger.info(f"Berhasil memuat data: {path_interim}")
        
        # 3. Cek Missing Value pakai fungsi utils kamu
        print("--- Ringkasan Missing Values ---")
        print(utils.check_missing_values(df_interim))
        
        return df_interim, cfg
    
    except Exception as e:
        logger.error(f"Gagal memuat data. Error: {e}")
        return None, cfg

# Eksekusi
df, config = load_and_validate()

2026-02-17 19:28:12,757 - INFO - Berhasil memuat data: ../data/interim/data_gabungan.pkl


--- Ringkasan Missing Values ---
          Jumlah  Persen (%)
pm10          68        4.06
pm25          97        5.79
so2          113        6.75
co            32        1.91
o3            64        3.82
no2           35        2.09
critical      16        0.96
category       1        0.06


### 3. Split Data (Skema Train-Valid-Test)

In [3]:
# --- 1. DEFINISI FUNGSI ---
def split_data(df_input, cfg):
    """
    Membagi data menjadi Train, Valid, dan Test.
    """
    target_col = cfg['label']
    
    # Hapus NaN pada target
    df_clean = df_input.dropna(subset=[target_col])
    
    # Filter kategori yang tidak diinginkan
    sebelum_filter = len(df_clean)
    df_clean = df_clean[df_clean[target_col] != 'TIDAK ADA DATA']
    
    logger.info(f"Berhasil membuang baris NaN dan {sebelum_filter - len(df_clean)} baris 'TIDAK ADA DATA'.")

    # Pisahkan Fitur dan Target
    X = df_clean.drop(columns=[target_col])
    y = df_clean[target_col]

    # Split Pertama (Train & Sisa)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, 
        test_size=(1 - cfg['params']['train_size']), 
        random_state=cfg['seed'], 
        stratify=y
    )

    # Split Kedua (Valid & Test)
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp, y_temp, 
        test_size=0.5, 
        random_state=cfg['seed'], 
        stratify=y_temp
    )
    
    logger.info(f"Splitting Sukses! Sisa kategori: {y.unique()}")
    
    # Ini adalah hasil yang "dikeluarkan" oleh fungsi
    return X_train, X_valid, X_test, y_train, y_valid, y_test

# --- 2. EKSEKUSI (MEMASAK) ---
# Di baris inilah hasil pembagian data disimpan ke dalam variabel-variabel
X_train, X_valid, X_test, y_train, y_valid, y_test = split_data(df, config)

# --- 3. TAMPILKAN HASIL (HASIL JADI) ---
print("-" * 30)
print(f"Jumlah Data Train: {len(X_train)}")
print(f"Jumlah Data Valid: {len(X_valid)}")
print(f"Jumlah Data Test : {len(X_test)}")
print(f"Kategori Target  : {y_train.unique()}")
print("-" * 30)

2026-02-17 19:28:12,805 - INFO - Berhasil membuang baris NaN dan 16 baris 'TIDAK ADA DATA'.
2026-02-17 19:28:12,826 - INFO - Splitting Sukses! Sisa kategori: <StringArray>
['SEDANG', 'BAIK', 'TIDAK SEHAT']
Length: 3, dtype: str


------------------------------
Jumlah Data Train: 1326
Jumlah Data Valid: 166
Jumlah Data Test : 166
Kategori Target  : <StringArray>
['BAIK', 'SEDANG', 'TIDAK SEHAT']
Length: 3, dtype: str
------------------------------


### 4. Simpan ke Folder Processed (.pkl)

In [4]:
def save_processed_data(X_train, X_valid, X_test, y_train, y_valid, y_test):
    """
    Menyimpan hasil split data ke folder processed menggunakan utils.serialize_data.
    """
    processed_path = '../data/processed/'
    
    # Simpan Fitur (X)
    utils.serialize_data(X_train, processed_path, 'X_train.pkl')
    utils.serialize_data(X_valid, processed_path, 'X_valid.pkl')
    utils.serialize_data(X_test, processed_path, 'X_test.pkl')
    
    # Simpan Target (y)
    utils.serialize_data(y_train, processed_path, 'y_train.pkl')
    utils.serialize_data(y_valid, processed_path, 'y_valid.pkl')
    utils.serialize_data(y_test, processed_path, 'y_test.pkl')
    
    logger.info("Tahap penyimpanan data processed selesai.")

# Eksekusi
save_processed_data(X_train, X_valid, X_test, y_train, y_valid, y_test)

2026-02-17 19:28:12,863 - INFO - Tahap penyimpanan data processed selesai.


SUCCESS: Object disimpan di ../data/processed/X_train.pkl
SUCCESS: Object disimpan di ../data/processed/X_valid.pkl
SUCCESS: Object disimpan di ../data/processed/X_test.pkl
SUCCESS: Object disimpan di ../data/processed/y_train.pkl
SUCCESS: Object disimpan di ../data/processed/y_valid.pkl
SUCCESS: Object disimpan di ../data/processed/y_test.pkl


### Blok 5: Proses Imputasi (Handling Missing Values)

In [5]:
def handle_imputation(X_train, X_valid, X_test, cfg):
    """
    Melakukan imputasi missing values menggunakan nilai Mean 
    yang dihitung hanya dari data Train.
    """
    # 1. PERBAIKAN: Tambahkan ['imputation'] sebelum ['impute_mean']
    cols_to_impute = cfg['imputation']['impute_mean']
    
    # Buat copy agar data asli tidak tertimpa
    X_train_imputed = X_train.copy()
    X_valid_imputed = X_valid.copy()
    X_test_imputed = X_test.copy()
    
    logger.info(f"Memulai imputasi untuk kolom: {cols_to_impute}")

    # 2. Proses per kolom
    for col in cols_to_impute:
        # HITUNG MEAN HANYA DARI TRAIN
        mean_value = X_train[col].mean()
        
        # Isi nilai NaN menggunakan mean dari Train
        X_train_imputed[col] = X_train_imputed[col].fillna(mean_value)
        X_valid_imputed[col] = X_valid_imputed[col].fillna(mean_value)
        X_test_imputed[col] = X_test_imputed[col].fillna(mean_value)
        
        logger.info(f"Kolom {col} diisi dengan nilai mean: {mean_value:.2f}")

    return X_train_imputed, X_valid_imputed, X_test_imputed

# --- EKSEKUSI ---
X_train_imputed, X_valid_imputed, X_test_imputed = handle_imputation(X_train, X_valid, X_test, config)

# Verifikasi:
print("\nCek Missing Values setelah Imputasi (X_train):")
# PERBAIKAN JUGA DISINI: Tambahkan ['imputation']
print(X_train_imputed[config['imputation']['impute_mean']].isnull().sum())

2026-02-17 19:28:12,880 - INFO - Memulai imputasi untuk kolom: ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
2026-02-17 19:28:12,883 - INFO - Kolom pm10 diisi dengan nilai mean: 51.36
2026-02-17 19:28:12,887 - INFO - Kolom pm25 diisi dengan nilai mean: 76.70
2026-02-17 19:28:12,891 - INFO - Kolom so2 diisi dengan nilai mean: 34.64
2026-02-17 19:28:12,896 - INFO - Kolom co diisi dengan nilai mean: 11.77
2026-02-17 19:28:12,900 - INFO - Kolom o3 diisi dengan nilai mean: 32.07
2026-02-17 19:28:12,902 - INFO - Kolom no2 diisi dengan nilai mean: 18.92



Cek Missing Values setelah Imputasi (X_train):
pm10    0
pm25    0
so2     0
co      0
o3      0
no2     0
dtype: int64


### Blok 6: Simpan Data yang Sudah Di-imputasi

In [6]:
def save_imputed_data(X_train, X_valid, X_test):
    """
    Menyimpan data hasil imputasi ke folder processed.
    """
    processed_path = '../data/processed/'
    
    utils.serialize_data(X_train, processed_path, 'X_train_imputed.pkl')
    utils.serialize_data(X_valid, processed_path, 'X_valid_imputed.pkl')
    utils.serialize_data(X_test, processed_path, 'X_test_imputed.pkl')
    
    logger.info("Data hasil imputasi berhasil diamankan!")

# Eksekusi
save_imputed_data(X_train_imputed, X_valid_imputed, X_test_imputed)

2026-02-17 19:28:12,938 - INFO - Data hasil imputasi berhasil diamankan!


SUCCESS: Object disimpan di ../data/processed/X_train_imputed.pkl
SUCCESS: Object disimpan di ../data/processed/X_valid_imputed.pkl
SUCCESS: Object disimpan di ../data/processed/X_test_imputed.pkl


### Blok 7: Label Encoding untuk Target

In [7]:
from sklearn.preprocessing import LabelEncoder

def handle_target_encoding(y_train, y_valid, y_test):
    """
    Mengubah target kategorikal menjadi angka menggunakan LabelEncoder.
    """
    le = LabelEncoder()
    
    # Fit (belajar) hanya dari data Train
    y_train_encoded = le.fit_transform(y_train)
    
    # Transform data Valid dan Test berdasarkan apa yang dipelajari dari Train
    y_valid_encoded = le.transform(y_valid)
    y_test_encoded = le.transform(y_test)
    
    # Simpan mappingnya biar kita tau angka 0 itu apa, 1 itu apa
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    logger.info(f"Target Encoding selesai. Mapping: {mapping}")
    
    return y_train_encoded, y_valid_encoded, y_test_encoded, le

# --- EKSEKUSI ---
y_train_encoded, y_valid_encoded, y_test_encoded, le_target = handle_target_encoding(y_train, y_valid, y_test)

2026-02-17 19:28:12,952 - INFO - Target Encoding selesai. Mapping: {'BAIK': np.int64(0), 'SEDANG': np.int64(1), 'TIDAK SEHAT': np.int64(2)}


### Blok 8: One-Hot Encoding untuk Fitur (stasiun & critical) & Scalling

In [8]:
def handle_feature_engineering(X_train, X_valid, X_test):
    """
    1. Feature Encoding (One-Hot)
    2. Feature Scaling (Standardization)
    """
    # --- PART A: ENCODING (Kode lama kamu) ---
    cols_to_drop = ['tanggal', 'max']
    X_train_sub = X_train.drop(columns=[c for c in cols_to_drop if c in X_train.columns])
    X_valid_sub = X_valid.drop(columns=[c for c in cols_to_drop if c in X_valid.columns])
    X_test_sub = X_test.drop(columns=[c for c in cols_to_drop if c in X_test.columns])

    cat_cols = X_train_sub.select_dtypes(include=['object']).columns.tolist()
    
    X_train_encoded = pd.get_dummies(X_train_sub, columns=cat_cols)
    X_valid_encoded = pd.get_dummies(X_valid_sub, columns=cat_cols)
    X_test_encoded = pd.get_dummies(X_test_sub, columns=cat_cols)
    
    X_train_encoded, X_valid_encoded = X_train_encoded.align(X_valid_encoded, join='left', axis=1, fill_value=0)
    X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

    # --- PART B: SCALING (TAMBAHAN BARU) ---
    logger.info("Melakukan Scaling pada fitur numerik...")
    scaler = StandardScaler()
    
    # Fit & Transform data Train
    X_train_final = pd.DataFrame(scaler.fit_transform(X_train_encoded), 
                                 columns=X_train_encoded.columns, 
                                 index=X_train_encoded.index)
    
    # Transform data Valid & Test (menggunakan parameter dari Train)
    X_valid_final = pd.DataFrame(scaler.transform(X_valid_encoded), 
                                 columns=X_valid_encoded.columns, 
                                 index=X_valid_encoded.index)
    
    X_test_final = pd.DataFrame(scaler.transform(X_test_encoded), 
                                columns=X_test_encoded.columns, 
                                index=X_test_encoded.index)
    
    # Simpan scaler ke folder models untuk deployment nanti
    joblib.dump(scaler, '../models/scaler.pkl')
    logger.info("Feature Engineering & Scaling selesai.")
    
    return X_train_final, X_valid_final, X_test_final

# --- EKSEKUSI ---
X_train_final, X_valid_final, X_test_final = handle_feature_engineering(X_train_imputed, X_valid_imputed, X_test_imputed)

2026-02-17 19:28:13,022 - INFO - Melakukan Scaling pada fitur numerik...
2026-02-17 19:28:13,037 - INFO - Feature Engineering & Scaling selesai.


### Blok 9: Final Touch & Saving Data

In [9]:
def save_final_preprocessed_data(X_train, X_valid, X_test, y_train, y_valid, y_test, le):
    """
    Menyimpan data pkl dan label encoder.
    """
    final_path = '../data/processed/'
    
    # Simpan Fitur & Target
    utils.serialize_data(X_train, final_path, 'X_train_final.pkl')
    utils.serialize_data(X_valid, final_path, 'X_valid_final.pkl')
    utils.serialize_data(X_test, final_path, 'X_test_final.pkl')
    utils.serialize_data(y_train, final_path, 'y_train_final.pkl')
    utils.serialize_data(y_valid, final_path, 'y_valid_final.pkl')
    utils.serialize_data(y_test, final_path, 'y_test_final.pkl')
    
    # Simpan Label Encoder
    joblib.dump(le, '../models/label_encoder.pkl')
    
    logger.info("SEMUA PROSES PREPROCESSING + SCALING SELESAI!")

# Eksekusi (Pastikan variabel 'le_target' dari Blok 7 kamu masukkan ke sini)
save_final_preprocessed_data(X_train_final, X_valid_final, X_test_final, 
                             y_train_encoded, y_valid_encoded, y_test_encoded, 
                             le_target)

2026-02-17 19:28:13,067 - INFO - SEMUA PROSES PREPROCESSING + SCALING SELESAI!


SUCCESS: Object disimpan di ../data/processed/X_train_final.pkl
SUCCESS: Object disimpan di ../data/processed/X_valid_final.pkl
SUCCESS: Object disimpan di ../data/processed/X_test_final.pkl
SUCCESS: Object disimpan di ../data/processed/y_train_final.pkl
SUCCESS: Object disimpan di ../data/processed/y_valid_final.pkl
SUCCESS: Object disimpan di ../data/processed/y_test_final.pkl
